# LeetCode 1288 — Remove Covered Intervals bằng Python

Link đề: https://leetcode.com/problems/remove-covered-intervals/ · Độ khó: **Medium** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def removeCoveredIntervals(self, intervals: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu chính xác định nghĩa **khoảng bị phủ**: `[a, b)` bị `[c, d)` phủ ⇔ `c ≤ a` và `b ≤ d`.
2. Viết lời giải **so mọi cặp `O(n²)`** làm chuẩn, rồi tối ưu bằng **sắp xếp + quét một lượt**.
3. Nắm mẹo **sắp xếp theo (start tăng, end giảm)** để xử lý các khoảng cùng điểm đầu, và kỹ thuật **theo dõi `max_end`**.
4. Biết thêm cách quét giữ "khoảng trước" và cách **bucket theo start** `O(n + V)` khi toạ độ nhỏ.
5. Tránh lỗi kinh điển: sort chỉ theo start, dùng `<` thay vì `≤`, so một khoảng với chính nó, chỉ so với khoảng liền trước.

## 2. Đề bài & ví dụ

Cho mảng `intervals`, mỗi phần tử `[l_i, r_i]` biểu diễn khoảng nửa mở `[l_i, r_i)`. Hãy **xoá mọi khoảng bị một khoảng khác trong danh sách phủ**, rồi trả về **số khoảng còn lại**.

Khoảng `[a, b)` bị khoảng `[c, d)` phủ khi và chỉ khi `c ≤ a` và `b ≤ d`.

**Ví dụ 1:** `intervals = [[1,4],[3,6],[2,8]]` → `2` (`[3,6]` bị `[2,8]` phủ).

**Ví dụ 2:** `intervals = [[1,4],[2,3]]` → `1`.

**Ràng buộc:** `1 <= intervals.length <= 1000`, `intervals[i].length == 2`, `0 <= l_i < r_i <= 10^5`, các khoảng **đôi một khác nhau**.

In [1]:
import random
import time
from typing import List

examples = [([[1, 4], [3, 6], [2, 8]], 2), ([[1, 4], [2, 3]], 1)]
for iv, want in examples:
    print(iv, "-> mong đợi", want)

[[1, 4], [3, 6], [2, 8]] -> mong đợi 2
[[1, 4], [2, 3]] -> mong đợi 1


## 3. Kiến thức nền

### 3.1 Kiểm tra "phủ" giữa hai khoảng

Viết thành hàm nhỏ để code rõ nghĩa. Chú ý dùng `<=` ở cả hai đầu (cho phép trùng đầu mút).

In [2]:
def covers(big, small):
    """big = [c, d) có phủ small = [a, b) không?"""
    return big[0] <= small[0] and small[1] <= big[1]

print(covers([2, 8], [3, 6]), covers([1, 4], [3, 6]), covers([1, 4], [1, 3]), covers([1, 3], [1, 4]))

True False True False


### 3.2 Sắp xếp theo nhiều tiêu chí bằng `key`

`sorted(a, key=lambda x: (x[0], -x[1]))`: so `x[0]` tăng dần, nếu bằng thì so `-x[1]` tăng dần ⇔ `x[1]` **giảm dần**. Tuple được so sánh từng thành phần từ trái sang.

In [3]:
iv = [[1, 4], [3, 6], [1, 8], [2, 8], [1, 5]]
print("theo start:        ", sorted(iv))
print("start ↑, end ↓:    ", sorted(iv, key=lambda x: (x[0], -x[1])))

theo start:         [[1, 4], [1, 5], [1, 8], [2, 8], [3, 6]]
start ↑, end ↓:     [[1, 8], [1, 5], [1, 4], [2, 8], [3, 6]]


### 3.3 Biến chạy `max_end`

Mẫu "duyệt và nhớ giá trị lớn nhất đã thấy": `best = max(best, x)`. Ở bài này, `max_end` là điểm cuối xa nhất của các khoảng đã xét.

In [4]:
ends = [4, 8, 6, 9, 3]
best = -1
for e in ends:
    best = max(best, e)
    print(f"gặp end={e}, max_end={best}")

gặp end=4, max_end=4
gặp end=8, max_end=8
gặp end=6, max_end=8
gặp end=9, max_end=9
gặp end=3, max_end=9


### 3.4 `enumerate` và so mọi cặp `i ≠ j`

Khi so mọi cặp, cần bỏ qua trường hợp `i == j` (một khoảng luôn "phủ" chính nó theo định nghĩa `≤`).

In [5]:
iv = [[1, 4], [2, 3]]
for i, a in enumerate(iv):
    for j, b in enumerate(iv):
        if i != j:
            print(f"{b} phủ {a}? {covers(b, a)}")

[2, 3] phủ [1, 4]? False
[1, 4] phủ [2, 3]? True


### 3.5 Mảng bucket theo toạ độ

Vì `0 ≤ l < r ≤ 10^5`, ta có thể tạo mảng `best_end[s]` = điểm cuối lớn nhất của các khoảng bắt đầu tại `s` — giống counting sort, không cần so sánh.

In [6]:
iv = [[1, 4], [3, 6], [2, 8], [1, 3]]
V = max(r for _, r in iv)
best_end = [-1] * (V + 1)
for l, r in iv:
    best_end[l] = max(best_end[l], r)
print({s: e for s, e in enumerate(best_end) if e != -1})

{1: 4, 2: 8, 3: 6}


## 4. Ý tưởng nền tảng / trực giác

Vẽ **Ví dụ 1** trên trục số:

```
      0 1 2 3 4 5 6 7 8
[1,4)   |-----|
[3,6)       |-----|
[2,8)     |-----------|
```

- `[3,6)` nằm trọn trong `[2,8)` ⇒ bị xoá.
- `[1,4)` bắt đầu sớm hơn `[2,8)` ⇒ không bị `[2,8)` phủ; cũng không bị `[3,6)` phủ ⇒ giữ.
- `[2,8)` không bị ai phủ ⇒ giữ. Còn **2**.

**Trực giác cho cách sắp xếp:** nếu xét các khoảng theo **điểm đầu tăng dần**, thì mọi khoảng đã xét trước đều có điểm đầu `≤` khoảng hiện tại. Khi đó khoảng hiện tại bị phủ ⇔ **có** một khoảng trước nó kết thúc **xa hơn hoặc bằng** ⇔ `end ≤ max_end` của các khoảng trước. Rắc rối duy nhất: hai khoảng **cùng điểm đầu** — khoảng dài hơn phải được xét **trước** để nó "phủ" khoảng ngắn hơn ⇒ sort `end` giảm dần khi `start` bằng nhau.

In [7]:
for iv, want in examples:
    kept = [a for i, a in enumerate(iv)
            if not any(covers(b, a) for j, b in enumerate(iv) if j != i)]
    print(iv, "-> còn lại", kept, "| số lượng", len(kept), "| mong đợi", want)

[[1, 4], [3, 6], [2, 8]] -> còn lại [[1, 4], [2, 8]] | số lượng 2 | mong đợi 2
[[1, 4], [2, 3]] -> còn lại [[1, 4]] | số lượng 1 | mong đợi 1


## 5. Cách 1 — Brute Force / baseline: so mọi cặp

**Ý tưởng:** với mỗi khoảng `a`, kiểm tra xem có khoảng `b` **khác** nào phủ nó không. Đếm số khoảng không bị phủ.

In [8]:
def remove_covered_brute(intervals):
    n = len(intervals)
    remaining = 0
    for i in range(n):
        a, b = intervals[i]
        covered = False
        for j in range(n):
            if i != j:
                c, d = intervals[j]
                if c <= a and b <= d:
                    covered = True
                    break
        if not covered:
            remaining += 1
    return remaining

for iv, want in examples:
    print(remove_covered_brute(iv), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


**Độ phức tạp:** `O(n²)` thời gian (`n ≤ 1000` ⇒ tối đa `10^6` phép so — vẫn qua LeetCode), `O(1)` bộ nhớ.

**Nút thắt:** mỗi khoảng phải so với **mọi** khoảng khác, kể cả những khoảng "rõ ràng không liên quan". Nếu `n` lên `10^5`, cách này hết thời gian. Sắp xếp giúp mỗi khoảng chỉ cần so với **một con số** (`max_end`).

In [9]:
rng0 = random.Random(0)
for n in [500, 1000, 2000]:
    iv = []
    seen = set()
    while len(iv) < n:
        l = rng0.randint(0, 10**5 - 1); r = rng0.randint(l + 1, 10**5)
        if (l, r) not in seen:
            seen.add((l, r)); iv.append([l, r])
    t = time.perf_counter()
    remove_covered_brute(iv)
    print(f"n={n}: brute force {time.perf_counter() - t:.3f}s")

n=500: brute force 0.001s
n=1000: brute force 0.002s
n=2000: brute force 0.006s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** sắp xếp theo `(start tăng, end giảm)`. Duyệt theo thứ tự đó, giữ `max_end` = điểm cuối lớn nhất của các khoảng đã duyệt.

**Quy tắc "nếu… thì…":**
- Nếu `end ≤ max_end` ⇒ khoảng hiện tại **bị phủ** (bởi khoảng đạt `max_end`, vốn có `start ≤` start hiện tại).
- Nếu `end > max_end` ⇒ **không ai trước** phủ nó; và **không ai sau** phủ được nó (khoảng sau có start lớn hơn, hoặc cùng start nhưng end nhỏ hơn) ⇒ **giữ**, cập nhật `max_end = end`.

**Invariant:** sau khi duyệt `i` khoảng đầu, `max_end` là end lớn nhất trong chúng, và `count` là số khoảng (trong `i` khoảng đó) không bị bất kỳ khoảng nào trong **toàn bộ** danh sách phủ.

Vì sao cần `end giảm` khi `start` bằng nhau? Ví dụ `[1,2]` và `[1,4]`: nếu `[1,2]` đi trước, nó được "giữ" (vì `2 > max_end`) dù thật ra bị `[1,4]` phủ.

## 7. Các cách tiếp theo

### Cách 2 — Sort (start ↑, end ↓) + theo dõi `max_end`

**Từng đoạn:**
1. `intervals.sort(key=lambda x: (x[0], -x[1]))` (hoặc sort trên bản sao).
2. `count = 0`, `max_end = -1` (nhỏ hơn mọi `r ≥ 1`).
3. Với mỗi `(l, r)`: nếu `r > max_end` thì `count += 1`, `max_end = r`.

In [10]:
def remove_covered_sort(intervals):
    count, max_end = 0, -1
    for l, r in sorted(intervals, key=lambda x: (x[0], -x[1])):
        if r > max_end:            # không bị khoảng nào trước phủ
            count += 1
            max_end = r
        # ngược lại: r <= max_end -> bị phủ, bỏ qua
    return count

for iv, want in examples:
    print(remove_covered_sort(iv), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


In [11]:
class SolutionSort:
    def removeCoveredIntervals(self, intervals: List[List[int]]) -> int:
        count, max_end = 0, -1
        for _, r in sorted(intervals, key=lambda x: (x[0], -x[1])):
            if r > max_end:
                count += 1
                max_end = r
        return count

print([SolutionSort().removeCoveredIntervals(iv) for iv, _ in examples])

[2, 1]


**Độ phức tạp Cách 2:** `O(n log n)` (sắp xếp) + `O(n)` quét; bộ nhớ `O(n)` cho bản sao đã sắp xếp (hoặc `O(1)` thêm nếu sort tại chỗ, chấp nhận sửa input).

### Cách 3 — Sort theo start, giữ "khoảng đang giữ cuối cùng"

**Ý tưởng:** sort mặc định (`start ↑`, `end ↑`). Giữ `(ps, pe)` = khoảng được giữ gần nhất.
- Nếu `r ≤ pe` ⇒ khoảng hiện tại bị phủ ⇒ bỏ.
- Nếu `l == ps` (và `r > pe`) ⇒ khoảng **trước** bị khoảng hiện tại phủ ⇒ **thay thế**: không tăng `count`, chỉ cập nhật `pe = r`.
- Ngược lại ⇒ khoảng mới được giữ: `count += 1`, `(ps, pe) = (l, r)`.

Cách này không cần mẹo `-end`, nhưng phải nghĩ kỹ trường hợp cùng start.

In [12]:
def remove_covered_prev(intervals):
    iv = sorted(intervals)                 # start ↑, end ↑
    count = 0
    ps, pe = -1, -1
    for l, r in iv:
        if r <= pe:
            continue                       # bị (ps, pe) phủ (ps <= l)
        if l == ps:
            pe = r                         # khoảng trước bị khoảng này phủ -> thay thế
        else:
            count += 1
            ps, pe = l, r
    return count

for iv, want in examples:
    print(remove_covered_prev(iv), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


Lưu ý điều kiện `r <= pe` dùng `pe` của khoảng đang giữ — là `max_end` thật sự, vì mỗi khi giữ khoảng mới thì `r > pe` cũ nên `pe` luôn tăng.

**Độ phức tạp Cách 3:** `O(n log n)` thời gian, `O(n)` bộ nhớ (bản sao).

### Cách 4 — Bucket theo start (không sort so sánh), `O(n + V)`

**Ý tưởng:**
1. Với mỗi điểm đầu `s`, chỉ khoảng có `end` **lớn nhất** tại `s` có thể sống sót (các khoảng khác cùng start bị nó phủ).
2. Quét `s` từ nhỏ tới lớn, giữ `max_end`. Khoảng đại diện `[s, best_end[s])` sống sót ⇔ `best_end[s] > max_end`.

In [13]:
def remove_covered_bucket(intervals):
    V = max(r for _, r in intervals)
    best_end = [-1] * (V + 1)
    for l, r in intervals:
        if r > best_end[l]:
            best_end[l] = r
    count, max_end = 0, -1
    for s in range(V + 1):
        e = best_end[s]
        if e > max_end:           # e = -1 (không có khoảng) không bao giờ > max_end >= -1
            count += 1
            max_end = e
    return count

for iv, want in examples:
    print(remove_covered_bucket(iv), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


**Độ phức tạp Cách 4:** `O(n + V)` thời gian với `V = max r ≤ 10^5`, `O(V)` bộ nhớ. Nhanh khi `n` lớn và toạ độ nhỏ; với `n ≤ 1000` thì Cách 2 thường gọn hơn.

## 8. TRACE từng bước

Trace Cách 2 trên `[[1,4],[3,6],[2,8],[1,2],[3,5]]`. Sau khi sort `(start ↑, end ↓)`: `[1,4], [1,2], [2,8], [3,6], [3,5]`.

| khoảng | max_end trước | r > max_end? | hành động | count | max_end sau |
|---|---|---|---|---|---|
| [1,4] | -1 | có | giữ | 1 | 4 |
| [1,2] | 4 | không | bị phủ (bởi [1,4]) | 1 | 4 |
| [2,8] | 4 | có | giữ | 2 | 8 |
| [3,6] | 8 | không | bị phủ (bởi [2,8]) | 2 | 8 |
| [3,5] | 8 | không | bị phủ | 2 | 8 |

In [14]:
def trace_sort(intervals):
    order = sorted(intervals, key=lambda x: (x[0], -x[1]))
    print("sau khi sort (start ↑, end ↓):", order)
    count, max_end, who = 0, -1, None
    for l, r in order:
        if r > max_end:
            count += 1
            print(f"[{l},{r}): r={r} > max_end={max_end} -> GIỮ, count={count}, max_end={r}")
            max_end, who = r, (l, r)
        else:
            print(f"[{l},{r}): r={r} <= max_end={max_end} -> bị [{who[0]},{who[1]}) phủ")
    return count

print("đáp án =", trace_sort([[1, 4], [3, 6], [2, 8], [1, 2], [3, 5]]))

sau khi sort (start ↑, end ↓): [[1, 4], [1, 2], [2, 8], [3, 6], [3, 5]]
[1,4): r=4 > max_end=-1 -> GIỮ, count=1, max_end=4
[1,2): r=2 <= max_end=4 -> bị [1,4) phủ
[2,8): r=8 > max_end=4 -> GIỮ, count=2, max_end=8
[3,6): r=6 <= max_end=8 -> bị [2,8) phủ
[3,5): r=5 <= max_end=8 -> bị [2,8) phủ
đáp án = 2


In [15]:
def trace_prev(intervals):
    iv = sorted(intervals)
    print("sau khi sort mặc định:", iv)
    count, ps, pe = 0, -1, -1
    for l, r in iv:
        if r <= pe:
            print(f"[{l},{r}): r <= pe={pe} -> bị [{ps},{pe}) phủ")
            continue
        if l == ps:
            print(f"[{l},{r}): cùng start với [{ps},{pe}) nhưng dài hơn -> THAY THẾ, count giữ {count}")
            pe = r
        else:
            count += 1
            ps, pe = l, r
            print(f"[{l},{r}): giữ mới, count={count}")
    return count

print("đáp án =", trace_prev([[1, 2], [1, 4], [3, 4], [2, 3]]))

sau khi sort mặc định: [[1, 2], [1, 4], [2, 3], [3, 4]]
[1,2): giữ mới, count=1
[1,4): cùng start với [1,2) nhưng dài hơn -> THAY THẾ, count giữ 1
[2,3): r <= pe=4 -> bị [1,4) phủ
[3,4): r <= pe=4 -> bị [1,4) phủ
đáp án = 1


## 9. Code nộp LeetCode

Chọn Cách 2: ngắn, rõ, `O(n log n)`. Không sửa input (dùng `sorted`).

In [16]:
class Solution:
    def removeCoveredIntervals(self, intervals: List[List[int]]) -> int:
        count = 0
        max_end = -1
        # start tăng; cùng start thì end giảm để khoảng dài đi trước
        for _, end in sorted(intervals, key=lambda x: (x[0], -x[1])):
            if end > max_end:
                count += 1
                max_end = end
        return count

for iv, want in examples:
    got = Solution().removeCoveredIntervals(iv)
    print(got, "OK" if got == want else "SAI")

2 OK
1 OK


## 10. Vì sao thuật toán đúng

Gọi thứ tự sau khi sort là `I_1, I_2, …, I_n` với `I_k = [l_k, r_k)`.

1. **Nếu `r_k ≤ max_end` thì `I_k` bị phủ.** `max_end = r_j` của một `I_j` với `j < k`. Do sort, `l_j ≤ l_k`. Vậy `l_j ≤ l_k` và `r_k ≤ r_j` ⇒ `I_j` phủ `I_k`. (`I_j ≠ I_k` vì các khoảng phân biệt.)
2. **Nếu `r_k > max_end` thì `I_k` không bị phủ.**
   - Mọi `j < k` có `r_j ≤ max_end < r_k` ⇒ không phủ được `I_k`.
   - Mọi `j > k`: hoặc `l_j > l_k` ⇒ không phủ được; hoặc `l_j = l_k` thì do sort end giảm, `r_j < r_k` (phân biệt) ⇒ không phủ được.
3. Vậy mỗi khoảng được phân loại đúng ⇒ `count` = số khoảng không bị phủ.

**Lưu ý:** "xoá các khoảng bị phủ" rồi đếm phần còn lại — có cần lo "khoảng phủ nó lại bị xoá"? Không: nếu `A` phủ `B` và `C` phủ `A` thì `C` cũng phủ `B` (tính bắc cầu), nên `B` vẫn bị một khoảng **còn lại** phủ. Đếm "không bị ai phủ" là đúng.

## 11. Phân tích Time / Space

Gọi `n = len(intervals)`, `V = max r_i ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. So mọi cặp | `O(n²)` (worst khi ít khoảng bị phủ; `break` sớm giúp khi nhiều khoảng bị phủ) | `O(1)` |
| 2. Sort (start ↑, end ↓) + max_end | `O(n log n)` | `O(n)` (bản sao) |
| 3. Sort + khoảng trước | `O(n log n)` | `O(n)` |
| 4. Bucket theo start | `O(n + V)` | `O(V)` |

## 12. Bộ test

Ví dụ đề, edge case (1 khoảng, cùng start, cùng end, lồng nhau nhiều tầng, chạm đầu mút, không giao nhau), đối chiếu ngẫu nhiên với brute force (seed cố định, các khoảng **phân biệt**, `0 ≤ l < r`), và kiểm tra không sửa input.

In [17]:
ALL = {
    "brute": remove_covered_brute, "sort": remove_covered_sort,
    "prev": remove_covered_prev, "bucket": remove_covered_bucket,
    "SolSort": lambda a: SolutionSort().removeCoveredIntervals(a),
    "Solution": lambda a: Solution().removeCoveredIntervals(a),
}
fixed = [
    ([[1, 4], [3, 6], [2, 8]], 2), ([[1, 4], [2, 3]], 1),
    ([[0, 1]], 1),
    ([[1, 2], [1, 4]], 1),                 # cùng start
    ([[1, 4], [3, 4]], 1),                 # cùng end
    ([[1, 10], [2, 9], [3, 8], [4, 7]], 1),  # lồng nhiều tầng
    ([[1, 3], [3, 5], [5, 7]], 3),         # chạm đầu mút, không phủ
    ([[1, 2], [3, 4], [5, 6]], 3),         # rời nhau
    ([[3, 10], [4, 10], [5, 11]], 2),
    ([[1, 2], [1, 4], [3, 4]], 1),
    ([[0, 100000], [1, 99999], [0, 1]], 1),
]
for iv, want in fixed:
    snapshot = [x[:] for x in iv]
    for name, f in ALL.items():
        got = f(iv)
        assert got == want, (name, iv, got, want)
    assert iv == snapshot
print("Test cố định: qua", len(fixed), "case, input không bị sửa")

Test cố định: qua 11 case, input không bị sửa


In [18]:
def random_intervals(n, V, rng):
    seen, res = set(), []
    while len(res) < n:
        l = rng.randint(0, V - 1)
        r = rng.randint(l + 1, V)
        if (l, r) not in seen:          # các khoảng phân biệt (đúng ràng buộc)
            seen.add((l, r))
            res.append([l, r])
    return res

rng = random.Random(1288)
for _ in range(1500):
    V = rng.choice([5, 10, 30, 1000])
    n = rng.randint(1, min(12, V * (V + 1) // 2))
    iv = random_intervals(n, V, rng)
    want = remove_covered_brute(iv)
    for name, f in ALL.items():
        assert f(iv) == want, (name, iv)
big = random_intervals(1000, 10**5, rng)
assert len({f(big) for f in ALL.values()}) == 1
print("1500 test ngẫu nhiên + 1 test n=1000: mọi cách khớp brute force")

1500 test ngẫu nhiên + 1 test n=1000: mọi cách khớp brute force


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sort chỉ theo start (end tăng) rồi dùng `max_end`

In [19]:
def wrong_sort_start_only(intervals):
    count, max_end = 0, -1
    for _, r in sorted(intervals):             # SAI: cùng start thì end ngắn đi trước
        if r > max_end:
            count += 1
            max_end = r
    return count

bad = [[1, 2], [1, 4]]
print("Code sai:", wrong_sort_start_only(bad), "| đúng là:", Solution().removeCoveredIntervals(bad))

Code sai: 2 | đúng là: 1


`[1,2]` đi trước nên được đếm, sau đó `[1,4]` cũng được đếm. **Code đúng:** `key=lambda x: (x[0], -x[1])` (Cách 2), hoặc xử lý `l == ps` như Cách 3.

### Lỗi 2 — Dùng `<` thay vì `<=` trong điều kiện phủ

In [20]:
def wrong_strict(intervals):
    n = len(intervals)
    remaining = 0
    for i in range(n):
        a, b = intervals[i]
        if not any(c < a and b < d for j, (c, d) in enumerate(intervals) if j != i):   # SAI
            remaining += 1
    return remaining

bad = [[1, 4], [1, 3], [2, 4]]
print("Code sai:", wrong_strict(bad), "| đúng là:", Solution().removeCoveredIntervals(bad))

Code sai: 3 | đúng là: 1


`[1,3]` và `[2,4]` đều bị `[1,4]` phủ (chung một đầu mút), nhưng `<` bỏ sót. **Code đúng:** `c <= a and b <= d`.

### Lỗi 3 — So mọi cặp nhưng quên `i != j`

In [21]:
def wrong_self(intervals):
    remaining = 0
    for a, b in intervals:
        if not any(c <= a and b <= d for c, d in intervals):     # SAI: tự phủ chính mình
            remaining += 1
    return remaining

for iv, want in examples:
    print("Code sai:", wrong_self(iv), "| đúng là:", want)

Code sai: 0 | đúng là: 2
Code sai: 0 | đúng là: 1


Mỗi khoảng đều "phủ" chính nó nên mọi khoảng bị coi là bị phủ ⇒ luôn trả 0. **Code đúng:** bỏ qua `j == i` (Cách 1).

### Lỗi 4 — Chỉ so với khoảng **liền trước** sau khi sort

In [22]:
def wrong_adjacent(intervals):
    iv = sorted(intervals, key=lambda x: (x[0], -x[1]))
    count = 1
    for k in range(1, len(iv)):
        pl, pr = iv[k - 1]
        l, r = iv[k]
        if not (pl <= l and r <= pr):           # SAI: chỉ xét khoảng liền trước
            count += 1
    return count

bad = [[1, 10], [2, 3], [4, 5]]
print("Code sai:", wrong_adjacent(bad), "| đúng là:", Solution().removeCoveredIntervals(bad))

Code sai: 2 | đúng là: 1


`[4,5]` không bị `[2,3]` phủ nhưng bị `[1,10]` (ở xa hơn) phủ. **Code đúng:** so với `max_end` của **mọi** khoảng trước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài khoảng (interval) ⇒ phản xạ đầu tiên: **sắp xếp theo điểm đầu**, rồi quét một lượt với một biến tổng hợp (`max_end`, `last_end`, …).
- Có quan hệ "chứa/phủ" ⇒ cần **tie-break**: cùng start thì khoảng **dài** đi trước (`-end`).
- Mẹo nhớ: "phủ" cần **hai** điều kiện; sort đảm bảo một điều kiện (`start`), biến chạy kiểm tra điều kiện còn lại (`end`).
- `n ≤ 1000` ⇒ `O(n²)` cũng qua; nhưng phỏng vấn sẽ hỏi `O(n log n)`.
- Toạ độ nhỏ (`≤ 10^5`) ⇒ có thể nghĩ tới bucket/counting thay cho sort.

## 15. Biến thể của bài

1. **Trả về danh sách khoảng còn lại** thay vì số lượng (Bài tập 3).
2. **Đếm khoảng "tối tiểu"** — không **chứa** khoảng nào khác: sort `(end ↑, start ↓)`, theo dõi `max_start` (Bài tập 5).
3. **LC 56 Merge Intervals:** gộp khoảng giao nhau — cũng sort theo start + `last_end`.
4. **LC 435 Non-overlapping Intervals:** số khoảng ít nhất cần xoá để không giao nhau — tham lam theo end.
5. **LC 452 Minimum Number of Arrows to Burst Balloons**, **LC 1235** (DP + binary search) — họ bài interval.
6. **Đếm số cặp (A, B) mà A phủ B** — cần Fenwick tree / sort + đếm, `O(n log n)`.

In [23]:
def merge_intervals(intervals):           # LC 56 — cùng mẫu "sort theo start + quét"
    res = []
    for l, r in sorted(intervals):
        if res and l <= res[-1][1]:
            res[-1][1] = max(res[-1][1], r)
        else:
            res.append([l, r])
    return res

print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))

[[1, 6], [8, 10], [15, 18]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So mọi cặp | mỗi khoảng kiểm tra mọi khoảng khác | `O(n²)` | `O(1)` | làm chuẩn, `n` nhỏ |
| 2. Sort (start ↑, end ↓) + max_end | bị phủ ⇔ `end ≤ max_end` | `O(n log n)` | `O(n)` | **nộp bài** |
| 3. Sort + khoảng trước | xử lý riêng cùng start (thay thế) | `O(n log n)` | `O(n)` | khi quên mẹo `-end` |
| 4. Bucket theo start | max end mỗi start, quét toạ độ | `O(n + V)` | `O(V)` | toạ độ nhỏ, `n` lớn |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `removeCoveredIntervals([[1,2],[1,4],[3,4]])` và `removeCoveredIntervals([[0,10],[5,12]])` bằng bao nhiêu?

**Bài 2 (tính tay):** Trace Cách 2 trên `[[3,10],[4,10],[5,11],[2,4]]`: ghi thứ tự sau khi sort và `max_end` sau từng bước.

**Bài 3 (điền TODO):** Hoàn thiện `remaining_intervals_todo(intervals)` trả về **danh sách** các khoảng còn lại (theo thứ tự đã sort).

**Bài 4 (tự viết lại):** Không nhìn mục 7, tự viết lại Cách 3 (sort mặc định + giữ khoảng trước).

**Bài 5 (mini challenge):** Viết `count_minimal(intervals)` đếm số khoảng **không chứa** bất kỳ khoảng nào khác trong danh sách, `O(n log n)`.

In [24]:
def remaining_intervals_todo(intervals):
    res = []
    max_end = -1
    for l, r in sorted(intervals, key=lambda x: (x[0], -x[1])):
        # TODO: nếu r > max_end thì thêm [l, r] vào res và cập nhật max_end
        pass
    return res

def count_minimal_todo(intervals):
    # TODO: sort theo (end tăng, start giảm), theo dõi max_start
    pass

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Vẽ trục số. (2) `[1,4)` phủ cả `[1,2)` và `[3,4)`. (3) `[0,10)` và `[5,12)` giao nhau nhưng không chứa nhau.

**Bài 2.** (1) Sort: `[2,4], [3,10], [4,10], [5,11]`. (2) `max_end`: `4 → 10 → 10 → 11`. (3) Giữ `[2,4]`, `[3,10]`, `[5,11]`; `[4,10]` bị phủ.

**Bài 3.** (1) Giống hệt Cách 2. (2) Thay `count += 1` bằng `res.append([l, r])`. (3) `len(res)` phải bằng đáp án Cách 2.

**Bài 4.** (1) `sorted(intervals)`. (2) Ba nhánh: `r <= pe` bỏ; `l == ps` thay `pe`; còn lại giữ mới. (3) Đối chiếu brute force.

**Bài 5.** (1) Đối xứng: `A` chứa `B` ⇔ `A.start ≤ B.start` và `B.end ≤ A.end`. (2) Sort theo end tăng (cùng end thì start **giảm** để khoảng ngắn hơn đi trước). (3) Khoảng hiện tại chứa một khoảng trước ⇔ `start ≤ max_start` của các khoảng trước; ngược lại nó tối tiểu.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().removeCoveredIntervals([[1, 2], [1, 4], [3, 4]]) == 1
assert Solution().removeCoveredIntervals([[0, 10], [5, 12]]) == 2

# Bài 2
assert sorted([[3, 10], [4, 10], [5, 11], [2, 4]], key=lambda x: (x[0], -x[1])) == \
    [[2, 4], [3, 10], [4, 10], [5, 11]]
assert Solution().removeCoveredIntervals([[3, 10], [4, 10], [5, 11], [2, 4]]) == 3

# Bài 3
def remaining_intervals_sol(intervals):
    res, max_end = [], -1
    for l, r in sorted(intervals, key=lambda x: (x[0], -x[1])):
        if r > max_end:
            res.append([l, r])
            max_end = r
    return res

assert remaining_intervals_sol([[1, 4], [3, 6], [2, 8]]) == [[1, 4], [2, 8]]

# Bài 4
def remove_covered_prev_sol(intervals):
    count, ps, pe = 0, -1, -1
    for l, r in sorted(intervals):
        if r <= pe:
            continue
        if l == ps:
            pe = r
        else:
            count += 1
            ps, pe = l, r
    return count

# Bài 5
def count_minimal_sol(intervals):
    count, max_start = 0, -1
    for l, r in sorted(intervals, key=lambda x: (x[1], -x[0])):
        if l > max_start:          # không chứa khoảng nào trước nó
            count += 1
        max_start = max(max_start, l)
    return count

def count_minimal_brute(intervals):
    return sum(1 for i, (a, b) in enumerate(intervals)
               if not any(a <= c and d <= b for j, (c, d) in enumerate(intervals) if j != i))

rng = random.Random(88)
for _ in range(1000):
    iv = random_intervals(rng.randint(1, 10), rng.choice([6, 20, 200]), rng)
    kept = remaining_intervals_sol(iv)
    assert len(kept) == remove_covered_brute(iv)
    assert all(not any(covers(b, a) for b in iv if b != a) for a in kept)
    assert remove_covered_prev_sol(iv) == remove_covered_brute(iv)
    assert count_minimal_sol(iv) == count_minimal_brute(iv)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "bị phủ" (sort + max_end):**

```python
count, max_end = 0, -1
for l, r in sorted(intervals, key=lambda x: (x[0], -x[1])):
    if r > max_end:      # không bị phủ
        count += 1
        max_end = r
return count
```

**Đối xứng "chứa khoảng khác":** sort `(end ↑, start ↓)`, theo dõi `max_start`.

**Checklist trước khi submit:**
- [ ] Sort theo start tăng, **cùng start thì end giảm**.
- [ ] Điều kiện phủ dùng `<=` ở cả hai đầu.
- [ ] So với `max_end` của **mọi** khoảng trước, không chỉ khoảng liền trước.
- [ ] Khởi tạo `max_end` nhỏ hơn mọi end (`-1`).
- [ ] Brute force bỏ qua `i == j`.
- [ ] Đúng chữ ký: `removeCoveredIntervals(self, intervals: List[List[int]]) -> int`.